In [1]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv


In [2]:
# 1. Load environment variables from .env file
try:
    load_dotenv(dotenv_path='/home/kiet/gkinhere/jobs-pipeline-analysis/.env')
    print("Environment variables loaded successfully.")
except Exception as e:
    print(f"Error loading environment variables: {e}")

# 2. Connect to the PostgreSQL database using environment variables
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = 'localhost' # tuy là chạy docker nhưng ở đây ko nằm trong network container nên ko dùng tên container
                      # và đây là máy host nên vẫn dùng localhost được
db_port = 5433 # port mình đã map bằng docker compose là 5433 và mình là máy host nên có thể dùng luôn
print(f"Connecting to database {db_name} at {db_host}:{db_port} as user {db_user}")

# 3. Establish the database connection
conn = None # Initialize connection variable
try:
    print("Connecting to the database...")
    conn = psycopg2.connect(
        dbname=db_name,
        user=db_user,
        password=db_password,
        host=db_host,
        port=db_port
    )
    print("Connection established successfully.")
except Exception as e:
    print(f"Error connecting to the database: {e}")

Environment variables loaded successfully.
Connecting to database topcvdb at localhost:5433 as user postgres
Connecting to the database...
Connection established successfully.


In [3]:
# Cell 3: Đọc dữ liệu từ PostgreSQL vào Pandas DataFrame

# Kiểm tra xem kết nối 'conn' từ Cell 2 có thực sự tồn tại và hoạt động không
if conn:
    # 1. Định nghĩa câu lệnh SQL (Query) để lấy dữ liệu
    query = "SELECT * FROM topcv_jobs_detailed WHERE status = 'completed' "
    print("Sending query to the database...")
    print(f"Query: {query}")
    
    # 2. Sử dụng hàm pd.read_sql() của Pandas để thực thi query và tải dữ liệu
    # Hàm này là "cây cầu thần kỳ" nối SQL và Pandas
    # Nó nhận vào câu lệnh SQL và đối tượng kết nối 'conn'
    df = pd.read_sql(query, conn)
    print("\n Data loaded successfully into DataFrame")
    # len(df) trả về số lượng hàng trong DataFrame
    print(f"Number of rows in DataFrame: {len(df)}")
    
    # 3. Đóng kết nối sau khi đã lấy xong dữ liệu. Đây là một thói quen tốt.
    conn.close()
    print("Database connection closed.")
    
    # 4. Hiển thị một vài dòng đầu tiên của DataFrame để kiểm tra dữ liệu
    print("\nFirst few rows of the DataFrame:")
    print(df.head())
else:
    print("No connection to the database. Cannot load data into DataFrame.")


Sending query to the database...
Query: SELECT * FROM topcv_jobs_detailed WHERE status = 'completed' 

 Data loaded successfully into DataFrame
Number of rows in DataFrame: 994
Database connection closed.

First few rows of the DataFrame:
   job_id source_site                                          job_title  \
0     196       topcv  Nhà Thiết Kế & Phát Triển Mẫu (Chuyên Về Thời ...   
1       7       topcv  Nhân Viên Tư Vấn Sản Phầm/Sales/CSKH/Thu Nhập ...   
2      11       topcv  Kế Toán Nội Bộ (Mức Lương Từ 11 - 15 Triệu/Thá...   
3      15       topcv  Nhân Viên Tư Vấn Chăm Sóc Khách Hàng - Lương C...   
4      20       topcv  Idol Live TikTok (Fulltime/Parttime) | Thu Nhậ...   

                                             job_url scrape_date     status  \
0  https://www.topcv.vn/viec-lam/nha-thiet-ke-pha...  2025-07-02  completed   
1  https://www.topcv.vn/viec-lam/nhan-vien-tu-van...  2025-07-02  completed   
2  https://www.topcv.vn/viec-lam/ke-toan-noi-bo-m...  2025-07-02  c

/tmp/ipykernel_67709/3335806790.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
print("\n 1. DataFrame Information:")
df.info()  # Hiển thị thông tin về DataFrame, bao gồm số lượng hàng, cột, kiểu dữ liệu, v.v.

print("\n 2. Null/NaN Values in DataFrame:")
missing_values = df.isnull().sum()  # Tính số lượng giá trị null/NaN trong mỗi cột
print(missing_values[missing_values > 0])  # Hiển thị chỉ những cột có giá trị null/NaN


 1. DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   job_id                     994 non-null    int64              
 1   source_site                994 non-null    object             
 2   job_title                  994 non-null    object             
 3   job_url                    994 non-null    object             
 4   scrape_date                994 non-null    object             
 5   status                     994 non-null    object             
 6   company_name_raw_list      994 non-null    object             
 7   salary_raw_list            994 non-null    object             
 8   location_raw_list          994 non-null    object             
 9   post_date_raw_list         994 non-null    object             
 10  company_name_detail        994 non-null    obj

In [5]:
print("[1]. Khám phá cột 'job_level'")
print(df['job_level'].value_counts())  # Hiển thị số lượng các giá trị duy nhất trong cột 'job_level'

print("\n[2]. Khám phá cột 'employment_type")
print(df['employment_type'].value_counts())  # Hiển thị số lượng các giá trị duy nhất trong cột 'employment_type'

print("\n[3]. Khám phá cột 'location_raw_list'")
print(df['location_raw_list'].value_counts().head(10))  # Hiển thị số lượng các giá trị duy nhất trong cột 'location_raw_list', chỉ lấy 10 giá trị đầu tiên

[1]. Khám phá cột 'job_level'
job_level
Nhân viên             846
Quản lý / Giám sát     50
Trưởng nhóm            39
Trưởng/Phó phòng       35
Giám đốc               15
Thực tập sinh           9
Name: count, dtype: int64

[2]. Khám phá cột 'employment_type
employment_type
Toàn thời gian    975
Bán thời gian      17
Thực tập            1
Khác                1
Name: count, dtype: int64

[3]. Khám phá cột 'location_raw_list'
location_raw_list
Hồ Chí Minh                  762
Hồ Chí Minh, Hà Nội           95
Hồ Chí Minh & 2 nơi khác      47
Hồ Chí Minh, Bình Dương       12
Hồ Chí Minh, Long An          11
Hồ Chí Minh & 3 nơi khác      11
Hồ Chí Minh & 4 nơi khác       7
Hồ Chí Minh & 5 nơi khác       7
Hồ Chí Minh, Bắc Ninh          4
Hồ Chí Minh & 12 nơi khác      4
Name: count, dtype: int64


In [6]:
# --- [4] Trả lời câu hỏi của bạn: "Vạch mặt" các list rỗng ---
print("\n\n--- [4] Kiểm tra các cột dạng list/mảng ---")
# Ta sẽ kiểm tra cột 'required_skills_tags'
# Dữ liệu đọc từ Postgres có thể là chuỗi "{}"
# Hoặc nếu scraper trả về list rỗng, nó có thể là []
num_empty_required_skills = df[df['required_skills_tags'].astype(str).isin(['[]', '{}'])].shape[0]
print(f"Số lượng job có danh sách 'Kỹ năng yêu cầu' bị rỗng ('[]' hoặc '{{}}'): {num_empty_required_skills}")

num_empty_preferred_skills = df[df['preferred_skills_tags'].astype(str).isin(['[]', '{}'])].shape[0]
print(f"Số lượng job có danh sách 'Kỹ năng nên có' bị rỗng ('[]' hoặc '{{}}'): {num_empty_preferred_skills}")



--- [4] Kiểm tra các cột dạng list/mảng ---
Số lượng job có danh sách 'Kỹ năng yêu cầu' bị rỗng ('[]' hoặc '{}'): 384
Số lượng job có danh sách 'Kỹ năng nên có' bị rỗng ('[]' hoặc '{}'): 734
